In [ ]:
# ============================================================
# CELL 1: COMPLETE COVERAGE ANALYZER (Minimal)
# ============================================================

import os
import re
import json
import time
import requests
from collections import defaultdict
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup

# --- CONFIG ---
BASE_URL = "https://user-guidance.analytical-platform.service.justice.gov.uk/"
BASE_DIR = os.path.join("..", "data")
TEXT_DIR = os.path.join(BASE_DIR, "text_chunks")
HEADERS = {"User-Agent": "Mozilla/5.0"}

# --- 1. DISCOVER URLs ---
def crawl_site(base_url, max_pages=300):
    visited, to_visit, urls = set(), [base_url], []
    domain = urlparse(base_url).netloc
    
    while to_visit and len(urls) < max_pages:
        url = to_visit.pop(0).split('#')[0].rstrip('/')
        if url in visited:
            continue
        visited.add(url)
        
        try:
            resp = requests.get(url, headers=HEADERS, timeout=10)
            if resp.status_code != 200:
                continue
            soup = BeautifulSoup(resp.text, 'html.parser')
            urls.append(url)
            
            for link in soup.find_all('a', href=True):
                full_url = urljoin(url, link['href']).split('#')[0].rstrip('/')
                if urlparse(full_url).netloc == domain and full_url not in visited:
                    if not any(x in full_url for x in ['.pdf', '.png', '.css', '.js']):
                        to_visit.append(full_url)
            
            if len(urls) % 20 == 0:
                print(f"Crawled {len(urls)} pages...")
            time.sleep(0.2)
        except:
            pass
    
    return urls

# --- 2. GET GROUND TRUTH ---
def get_page_text(soup):
    clone = BeautifulSoup(str(soup), 'html.parser')
    for tag in clone(['script', 'style', 'nav', 'footer', 'header', 'noscript']):
        tag.decompose()
    return set(re.sub(r'[^a-z0-9\s]', ' ', clone.get_text().lower()).split())

# --- 3. LOAD EXTRACTED TEXT ---
def load_extracted(url, text_dir):
    texts = []
    if not os.path.exists(text_dir):
        return set()
    
    for f in os.listdir(text_dir):
        if f.endswith('.txt') and '.metadata.' not in f:
            meta_path = os.path.join(text_dir, f"{f}.metadata.json")
            if os.path.exists(meta_path):
                try:
                    meta = json.load(open(meta_path))
                    if 'metadataAttributes' in meta:
                        meta = meta['metadataAttributes']
                    if meta.get('page_url') == url:
                        texts.append(open(os.path.join(text_dir, f)).read())
                except:
                    pass
    
    return set(re.sub(r'[^a-z0-9\s]', ' ', ' '.join(texts).lower()).split())

# --- 4. ANALYZE ---
def analyze(urls, text_dir):
    results = []
    
    for i, url in enumerate(urls):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=10)
            soup = BeautifulSoup(resp.text, 'html.parser')
            
            truth = get_page_text(soup)
            extracted = load_extracted(url, text_dir)
            matched = truth & extracted
            
            coverage = len(matched) / len(truth) * 100 if truth else 100
            results.append({'url': url, 'coverage': round(coverage, 1), 
                           'missed': len(truth) - len(matched)})
            
            if (i + 1) % 20 == 0:
                print(f"Analyzed {i + 1}/{len(urls)}")
        except:
            results.append({'url': url, 'coverage': 0, 'missed': 0})
    
    return results

# --- 5. REPORT ---
def report(results):
    successful = [r for r in results if r['coverage'] > 0]
    avg_coverage = sum(r['coverage'] for r in successful) / len(successful)
    total_missed = sum(r['missed'] for r in successful)
    
    print("\n" + "=" * 60)
    print("COVERAGE REPORT")
    print("=" * 60)
    print(f"Pages Analyzed:    {len(successful)}")
    print(f"Average Coverage:  {avg_coverage:.1f}%")
    print(f"Total Words Missed: {total_missed:,}")
    
    print("\n--- WORST PAGES ---")
    for r in sorted(successful, key=lambda x: x['coverage'])[:10]:
        print(f"  {r['coverage']:5.1f}%  |  {r['url']}")
    
    print("\n--- COVERAGE DISTRIBUTION ---")
    for label, low, high in [("Excellent 95-100%", 95, 101), ("Good 85-95%", 85, 95), 
                              ("Fair 70-85%", 70, 85), ("Poor <70%", 0, 70)]:
        count = len([r for r in successful if low <= r['coverage'] < high])
        print(f"  {label}: {count} pages")
    
    return avg_coverage

# --- RUN ---
print("Step 1: Crawling site...")
urls = crawl_site(BASE_URL)
print(f"Found {len(urls)} URLs\n")

print("Step 2: Analyzing coverage...")
results = analyze(urls, TEXT_DIR)

print("Step 3: Generating report...")
coverage = report(results)

# Save results
with open(os.path.join(BASE_DIR, "coverage_results.json"), 'w') as f:
    json.dump(results, f, indent=2)
print(f"\n✓ Results saved to {BASE_DIR}/coverage_results.json")

In [ ]:
# ============================================================
# CELL: INSPECT MISSING CONTENT FOR A URL
# ============================================================

import os
import re
import json
import requests
from bs4 import BeautifulSoup

# --- CONFIG ---
BASE_DIR = os.path.join("..", "data")
TEXT_DIR = os.path.join(BASE_DIR, "text_chunks")
HEADERS = {"User-Agent": "Mozilla/5.0"}

def inspect_missing(url):
    """Show what content is missing for a specific URL"""
    
    # 1. Fetch page
    resp = requests.get(url, headers=HEADERS, timeout=10)
    soup = BeautifulSoup(resp.text, 'html.parser')
    
    # 2. Get ground truth (all visible text)
    clone = BeautifulSoup(str(soup), 'html.parser')
    for tag in clone(['script', 'style', 'nav', 'footer', 'header', 'noscript']):
        tag.decompose()
    truth_words = set(re.sub(r'[^a-z0-9\s]', ' ', clone.get_text().lower()).split())
    truth_words = {w for w in truth_words if len(w) > 2}  # Skip tiny words
    
    # 3. Load extracted text
    extracted_texts = []
    for f in os.listdir(TEXT_DIR):
        if f.endswith('.txt') and '.metadata.' not in f:
            meta_path = os.path.join(TEXT_DIR, f"{f}.metadata.json")
            if os.path.exists(meta_path):
                try:
                    meta = json.load(open(meta_path))
                    if 'metadataAttributes' in meta:
                        meta = meta['metadataAttributes']
                    if meta.get('page_url') == url:
                        extracted_texts.append(open(os.path.join(TEXT_DIR, f)).read())
                except:
                    pass
    
    extracted_words = set(re.sub(r'[^a-z0-9\s]', ' ', ' '.join(extracted_texts).lower()).split())
    extracted_words = {w for w in extracted_words if len(w) > 2}
    
    # 4. Find missing words
    missed_words = truth_words - extracted_words
    matched_words = truth_words & extracted_words
    
    # 5. Find which elements contain missing content
    missing_by_element = {}
    body = clone.find("main") or clone.body or clone
    
    for element in body.find_all(['p', 'li', 'td', 'th', 'h1', 'h2', 'h3', 'h4', 
                                   'blockquote', 'aside', 'figcaption', 'dt', 'dd']):
        text = element.get_text(strip=True).lower()
        words = set(re.sub(r'[^a-z0-9\s]', ' ', text).split())
        overlap = words & missed_words
        
        if overlap:
            tag_name = element.name
            classes = element.get('class', [])
            key = f"<{tag_name}>" if not classes else f"<{tag_name} class='{' '.join(classes[:2])}'>"
            
            if key not in missing_by_element:
                missing_by_element[key] = {
                    'sample_text': text[:200],
                    'missed_words': list(overlap)
                }
    
    # 6. Print report
    coverage = len(matched_words) / len(truth_words) * 100 if truth_words else 100
    
    print("=" * 70)
    print(f"MISSING CONTENT REPORT")
    print("=" * 70)
    print(f"URL: {url}")
    print(f"Coverage: {coverage:.1f}%")
    print(f"Total words on page: {len(truth_words)}")
    print(f"Words extracted: {len(matched_words)}")
    print(f"Words missed: {len(missed_words)}")
    
    print("\n" + "-" * 50)
    print("MISSED WORDS (sample):")
    print("-" * 50)
    print(', '.join(list(missed_words)[:50]))
    
    print("\n" + "-" * 50)
    print("MISSING CONTENT BY ELEMENT:")
    print("-" * 50)
    
    for elem, data in missing_by_element.items():
        print(f"\n{elem}")
        print(f"  Sample:" "{data['sample_text'][:100]}...")
        print(f"  Missed: {', '.join(data['missed_words'][:10])}")
    
    return {
        'url': url,
        'coverage': round(coverage, 1),
        'missed_words': list(missed_words),
        'missing_elements': missing_by_element
    }




In [ ]:
# --- RUN ---
# Change this URL to inspect any page
url = "https://user-guidance.analytical-platform.service.justice.gov.uk/tools/create-a-derived-table/style-guide"
result = inspect_missing(url)